# Quickstart

This notebook builds, queries and updates a small map bank. It uses
`SyntheticGenerator`, a fast stand-in for a real map code, so it needs no GPU.
With the default `IPMGenerator` the same calls make real IPM maps.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

from adaptive_microlensing import (
    BankConfig,
    GeneratorSpec,
    MapBank,
    MapSpec,
    StoppingCriteria,
    SyntheticGenerator,
    VariabilitySpec,
    hit_summary,
)

generator = SyntheticGenerator()
config = BankConfig(
    variability=VariabilitySpec(map=MapSpec(4.0, 0.1), window_half_length=1.0, n_bin_edges=12),
    bank_map=MapSpec(2.0, 0.05),
    n_bin_edges=12,
    generator=GeneratorSpec.from_generator(generator),
)
path = Path(tempfile.mkdtemp()) / "bank"

## Build

Each region (image type) is sampled adaptively, then finalized, which freezes the
bin edges of its magnification probability distributions (MPDs).

In [ ]:
with MapBank.create(path, config) as bank:
    for region in ("minima", "saddle", "maxima"):
        summary = bank.build(region, StoppingCriteria(max_valid_points=40))
        bank.finalize(region)
        print(summary.region, summary.n_valid, summary.stop_reason)
    print(bank.summary())

## Query

A read-only bank answers queries without making maps.

In [ ]:
bank = MapBank.open(path)
result = bank.query(0.3, 0.2, 0.5)
print(result.status, result.region, result.interpolated_distance, result.threshold)
if result.is_hit:
    print(result.entry.kappa, result.entry.gamma, result.entry.s, result.entry.load().shape)
bank.close()

## Update

`fetch_many` returns a matching map for every row, making and adding a map on each miss.
Running the same queries again mostly finds the maps the first pass made. A few can
miss again, because every new map changes the tetrahedra around it.

In [ ]:
rng = np.random.default_rng(1)
queries = pd.DataFrame(
    {
        "kappa": rng.uniform(0.05, 2.0, 200),
        "gamma": rng.uniform(0.05, 2.0, 200),
        "s": rng.uniform(0.01, 0.99, 200),
    }
)
with MapBank.open(path, writable=True) as bank:
    first = bank.fetch_many(queries)
    second = bank.fetch_many(queries)
    print(first["fetch_status"].value_counts())
    print(second["fetch_status"].value_counts())
    print(bank.summary())
hit_summary(first)